# Step 15: Production Features

        _Learner Notebook_  
        Estimated time: **75 minutes**  
        Difficulty: **Advanced**

        ## Learning Objectives
        - [ ] Add structured logging around agent calls.
- [ ] Collect lightweight metrics and cost estimates.
- [ ] Trace prompts and responses for debugging.
- [ ] Close the course with a production-minded operating posture.

        ## Prerequisites
        - Step 14 completed.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Summary & Key Takeaways
8. Additional Resources


## Setup & Imports

The next cell adds the repository root to `sys.path` and imports the shared course helpers.
Most notebooks use the same bootstrap so the lesson content can stay focused on the concept,
not on path setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

Production readiness is about visibility. This notebook adds the minimum observability you need to reason about behavior, latency, failures, and rough usage costs.

Expected output and validation notes:

Expected output snapshot:

- The live answer is printed
- Response time metrics are summarized
- A small dashboard dictionary is produced


## Part 2: Core Implementation

### Instrumentation helpers


In [ ]:
import logging
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime
from time import perf_counter

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
)
logger = logging.getLogger("course.production")

@dataclass
class MetricPoint:
    name: str
    value: float
    timestamp: str

class MetricsCollector:
    def __init__(self):
        self.points = defaultdict(list)

    def record(self, name: str, value: float) -> None:
        self.points[name].append(
            MetricPoint(name=name, value=value, timestamp=datetime.utcnow().isoformat())
        )

    def summary(self, name: str) -> dict:
        values = [point.value for point in self.points[name]]
        return {
            "count": len(values),
            "min": min(values) if values else None,
            "max": max(values) if values else None,
            "avg": sum(values) / len(values) if values else None,
        }


## Part 2: Core Implementation

### Instrument a live run


In [ ]:
metrics = MetricsCollector()
production_agent = create_agent(
    name="ProductionAgent",
    instructions="Answer clearly and briefly for observability demos.",
)

start = perf_counter()
response = await production_agent.run("List three signals that help debug an agent system.")
duration = perf_counter() - start
metrics.record("response_time_seconds", duration)

logger.info("agent_call_completed", extra={"duration_seconds": duration})
print(response.text)
print_json(metrics.summary("response_time_seconds"), title="Metric summary")


## Part 3: Hands-On Exercises

Add a tiny cost estimator that multiplies total tokens by a placeholder rate and prints a dashboard dictionary.

Try the exercise yourself before looking at the solutions in Part 4.


In [ ]:
# TODO: estimate cost from the response usage or a simple placeholder token count


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
usage = getattr(response, "usage", None)
estimated_tokens = getattr(usage, "total_tokens", None) if usage else len(response.text.split()) * 2
estimated_cost = estimated_tokens * 0.000002
dashboard = {
    "response_time_seconds": duration,
    "estimated_tokens": estimated_tokens,
    "estimated_cost_usd": round(estimated_cost, 6),
}
print_json(dashboard, title="Production dashboard")


## Part 5: Best Practices & Tips

        - Log the events you will wish you had during incident review.
- Record latency and token usage close to the call boundary.
- Keep debugging utilities simple enough to run inside a notebook first.


## Summary & Key Takeaways

You closed the course with the core production concerns: logging, metrics, debugging, and rough cost awareness.


## Additional Resources

        - `helpers/debug.py`
- `scripts/smoke_live.py`
